# 事件回放器无干预身份门

新 `src/backtest/stage1_event_episode.py` 只加入一次分钟事件成交入口。本卡使用封存逐层点时f696+b472模型与Qlib账户，先对 `event=None` 的完整1056日回放核对逐股、现金、费、NAV、风险状态、订单、目标与模型训练时钟；零新拟合、零经济干预或候选账户。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"]="1"
import hashlib,json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from src.core.recorder_sources import load_verified_recorder_objects
from src.backtest.stage1_event_episode import replay_event_full_path
from src.workflow.stage1_dated_controller import DatedModel,DatedStage1Controllers
from src.workflow.stage1_f696_training_reference import prepare as prepare_f696_source
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
root=Path.cwd();card=root/"research/stage1_event_episode_noop_parity_20261029"
raw=(card/"plan.json").read_bytes();plan=json.loads(raw);sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert sha(root/plan["replay_code"])==plan["replay_code_sha256"]
parent=root/plan["parent_card"];old=root/plan["old_source_card"]
assert sha(parent/"summary.json")==plan["parent_summary_sha256"] and sha(parent/"verification_manifest.json")==plan["parent_manifest_sha256"] and sha(old/"plan.json")==plan["old_source_plan_sha256"]
parent_plan=json.loads((parent/"plan.json").read_text());old_plan=json.loads((old/"plan.json").read_text())
prepared=prepare_f696_source(root=root,card=old,plan=old_plan)
qlib.init(provider_uri=str(root/parent_plan["provider_uri"]),region="us",kernels=1)
names=["analysis/native/dated_f696_0","analysis/controller_audit",*[f"models/open_{year}" for year in parent_plan["signal_years"]],*[f"models/recovery_{year}" for year in (2020,2021,2022)]]
objects,receipt=load_verified_recorder_objects(parent,expected_recorder_id=plan["parent_recorder_id"],artifacts=names,expected_manifest_sha256=plan["parent_manifest_sha256"])
book=objects["analysis/native/dated_f696_0"];frame=prepared["frame"]
assert len(frame)==plan["expected_sessions"] and len(book["authority"])==len(frame) and book["audit"]["status"]=="passed"
open_models={year:DatedModel(objects[f"models/open_{year}"],f"{year-1}-12-31",f"{parent_plan['action_recorder_id']}:open_action_value_models/{year}") for year in parent_plan["signal_years"]}
recovery_models={}
for year in parent_plan["signal_years"]:
    if year in parent_plan["explicit_no_recovery_model_years"]:
        recovery_models[year]=DatedModel(None,f"{year-1}-12-31",f"explicit_no_model_{year}_insufficient_prior_teacher",explicit_no_model=True)
    else:
        recovery_models[year]=DatedModel(objects[f"models/recovery_{year}"],f"{year-1}-12-31",f"{plan['parent_recorder_id']}:models/recovery_{year}")
controller=DatedStage1Controllers(calendar=frame[["signal_date","trade_date"]],open_models=open_models,recovery_models=recovery_models)
pd.testing.assert_frame_equal(controller.audit(),objects["analysis/controller_audit"],check_dtype=False)
recovery_spec=IntradayRecoveryDirectAllocationSpec(root=root,**prepared["action_plan"]["recovery_spec"])
action_spec=Stage1OpenActionValueSpec(root=root,**prepared["action_plan"]["action_spec"])
replay=replay_event_full_path(frame,models=None,plan=parent_plan,slippage_bps=0,controller_by_date=controller,recovery_spec=recovery_spec,action_spec=action_spec,selection=prepared["selection"],event=None)
expected=book["independent_replay"]
pd.testing.assert_frame_equal(replay["replay"].reset_index(drop=True),expected.reset_index(drop=True),check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
pd.testing.assert_frame_equal(replay["orders"].reset_index(drop=True),book["orders"].reset_index(drop=True),check_dtype=False,rtol=0,atol=plan["maximum_currency_error_usd"])
pd.testing.assert_frame_equal(replay["targets"].reset_index(drop=True),book["targets"].reset_index(drop=True),check_dtype=False,rtol=0,atol=plan["maximum_weight_error"])
assert int(replay["replay"].recovery_action.sum())==plan["expected_recovery_days"]
fee_peak=max(replay["monthly_fee_shares"].values(),default=0.)
assert abs(fee_peak-book["audit"]["maximum_monthly_fee_counted_raw_shares"])<1e-7
monetary=["native_expected_nav","ending_soxl_quantity","ending_soxx_quantity","ending_cash","strategy_return","fees"]
errors={column:float((replay["replay"][column]-expected[column]).abs().max()) for column in monetary}
audit={"experiment":plan["experiment"],"plan_sha256":hashlib.sha256(raw).hexdigest(),"status":"passed_noop_replay_parity","source_recorder_id":plan["parent_recorder_id"],"source_readback_entries":len(receipt),"sessions":len(replay["replay"]),"opening_action_days":int(replay["replay"].execute_open_target.sum()),"recovery_days":int(replay["replay"].recovery_action.sum()),"orders":len(replay["orders"]),"controller_rows":len(controller.audit()),"max_errors":errors,"monthly_fee_peak":fee_peak,"new_model_fits":0,"new_native_accounts":0,"new_economic_event_branches":0,"historical_research_passed":False,"decision":"event_hook_noop_parity_pass_needs_checkpoint_and_event_branch"}
(card/"audit.json").write_text(json.dumps(audit,ensure_ascii=False,indent=2)+"\n")
print(json.dumps({"decision":audit["decision"],"sessions":audit["sessions"],"orders":audit["orders"],"recovery_days":audit["recovery_days"],"max_errors":errors},ensure_ascii=False,indent=2))

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[31693:MainThread](2026-10-29 11:58:20,705) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31693:MainThread](2026-10-29 11:58:20,707) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31693:MainThread](2026-10-29 11:58:20,707) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


[31693:MainThread](2026-10-29 11:58:20,792) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[31693:MainThread](2026-10-29 11:58:20,792) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[31693:MainThread](2026-10-29 11:58:20,793) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/full_path_tiered_allocation_20261013')}


{
  "decision": "event_hook_noop_parity_pass_needs_checkpoint_and_event_branch",
  "sessions": 1056,
  "orders": 1416,
  "recovery_days": 10,
  "max_errors": {
    "native_expected_nav": 0.0,
    "ending_soxl_quantity": 0.0,
    "ending_soxx_quantity": 0.0,
    "ending_cash": 0.0,
    "strategy_return": 0.0,
    "fees": 0.0
  }
}


## 中文结论

新单次分钟事件回放器在 `event=None` 下，使用逐层点时f696+b472的1056日、936个开盘执行、10个恢复、1416笔原订单，与封存原生Qlib父本的股数、现金、费用、NAV、风险状态、目标和订单逐项最大差0。决定 `event_hook_noop_parity_pass_needs_checkpoint_and_event_branch`。零新模型、零经济干预、零候选账户。

这是无事件身份门；尚未证明有事件后20会话自身状态续跑。当前引擎每次从起点回放，不能直接批量生成283513个分钟标签。另有2项聚焦合约检查通过：错误分钟时钟被拒绝，合成单笔SOXX事件只改变事件分支。真实历史事件和半日市尚未认证。下一门是可恢复checkpoint和单次事件的全历史／快照续跑一致性；不通过就停止。详情见 `audit.json`、`conclusion.md`。